# Macro & Market Relationships

How macroeconomic indicators relate to market performance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Load the analytics metrics (market data)
market_df = pd.read_csv('../data/processed/analytics_metrics.csv', parse_dates=['date'])

# Load the macro indicators
macro_df = pd.read_csv('../data/raw/macro_indicators_raw.csv', parse_dates=['date'])

print("Market data shape:", market_df.shape)
print("Macro data shape: ", macro_df.shape)
print("\nMarket columns:", list(market_df.columns))
print("Macro columns: ", list(macro_df.columns))
print("\nUnique macro indicators:", macro_df['indicator_name'].unique())

## 1. Macro Indicator Trends Over Time

In [ ]:
plt.rcParams['figure.figsize'] = (14, 5)

indicators = [
    "Federal Funds Rate",
    "10-Year Treasury Yield",
    "CPI (Inflation Proxy)",
    "Unemployment Rate",
]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for i, name in enumerate(indicators):
    ax = axes[i]
    subset = macro_df[macro_df['indicator_name'] == name].sort_values('date')

    ax.plot(subset['date'], subset['value'], linewidth=1.8, color='steelblue')
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.set_xlabel('Date', fontsize=10)
    ax.set_ylabel('Value', fontsize=10)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator(2))
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

    # Remove top and right spines
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Macro Indicator Trends Over Time', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 2. Federal Funds Rate vs SPY Cumulative Return

In [ ]:
# --- Prepare SPY monthly cumulative return ---
spy_daily = market_df[market_df['ticker'] == 'SPY'][['date', 'cumulative_return']].copy()
spy_daily = spy_daily.sort_values('date')
# Resample to month-end mean
spy_monthly = (
    spy_daily.set_index('date')
    .resample('ME')['cumulative_return']
    .mean()
    .reset_index()
)

# --- Prepare Federal Funds Rate monthly ---
fed_rate = (
    macro_df[macro_df['indicator_name'] == 'Federal Funds Rate']
    .sort_values('date')[['date', 'value']]
    .copy()
)
fed_rate_monthly = (
    fed_rate.set_index('date')
    .resample('ME')['value']
    .mean()
    .reset_index()
    .rename(columns={'value': 'fed_rate'})
)

# --- Merge on month-end date using merge_asof ---
spy_monthly = spy_monthly.sort_values('date')
fed_rate_monthly = fed_rate_monthly.sort_values('date')
merged = pd.merge_asof(spy_monthly, fed_rate_monthly, on='date', direction='nearest')

# --- Plot dual-axis chart ---
fig, ax1 = plt.subplots(figsize=(14, 5))

ax1.plot(merged['date'], merged['cumulative_return'] * 100,
         color='steelblue', linewidth=2, label='SPY Cumulative Return (%)')
ax1.set_xlabel('Date', fontsize=11)
ax1.set_ylabel('SPY Cumulative Return (%)', color='steelblue', fontsize=11)
ax1.tick_params(axis='y', labelcolor='steelblue')
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

ax2 = ax1.twinx()
ax2.plot(merged['date'], merged['fed_rate'],
         color='red', linewidth=2, linestyle='--', label='Federal Funds Rate')
ax2.set_ylabel('Federal Funds Rate (%)', color='red', fontsize=11)
ax2.tick_params(axis='y', labelcolor='red')
ax2.spines['top'].set_visible(False)

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=10)

ax1.set_title('Federal Funds Rate vs SPY Cumulative Return', fontsize=13, fontweight='bold')
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax1.xaxis.set_major_locator(mdates.YearLocator(2))
plt.setp(ax1.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 3. 10-Year Treasury Yield vs Tech Stocks Average Return

In [ ]:
tech_tickers = ['AAPL', 'MSFT', 'NVDA', 'AMZN', 'GOOGL']

# --- Average cumulative return for tech tickers, resampled monthly ---
tech_daily = market_df[market_df['ticker'].isin(tech_tickers)][['date', 'cumulative_return']].copy()
tech_daily = tech_daily.sort_values('date')
tech_monthly = (
    tech_daily.set_index('date')
    .resample('ME')['cumulative_return']
    .mean()
    .reset_index()
    .rename(columns={'cumulative_return': 'avg_cum_return'})
)

# --- 10-Year Treasury Yield monthly ---
treasury = (
    macro_df[macro_df['indicator_name'] == '10-Year Treasury Yield']
    .sort_values('date')[['date', 'value']]
    .copy()
)
treasury_monthly = (
    treasury.set_index('date')
    .resample('ME')['value']
    .mean()
    .reset_index()
    .rename(columns={'value': 'treasury_yield'})
)

# --- Merge ---
tech_monthly = tech_monthly.sort_values('date')
treasury_monthly = treasury_monthly.sort_values('date')
merged_tech = pd.merge_asof(tech_monthly, treasury_monthly, on='date', direction='nearest')

# --- Plot dual-axis chart ---
fig, ax1 = plt.subplots(figsize=(14, 5))

ax1.plot(merged_tech['date'], merged_tech['avg_cum_return'] * 100,
         color='steelblue', linewidth=2, label='Tech Avg Cumulative Return (%)')
ax1.set_xlabel('Date', fontsize=11)
ax1.set_ylabel('Avg Cumulative Return (%)', color='steelblue', fontsize=11)
ax1.tick_params(axis='y', labelcolor='steelblue')
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

ax2 = ax1.twinx()
ax2.plot(merged_tech['date'], merged_tech['treasury_yield'],
         color='red', linewidth=2, linestyle='--', label='10-Year Treasury Yield')
ax2.set_ylabel('10-Year Treasury Yield (%)', color='red', fontsize=11)
ax2.tick_params(axis='y', labelcolor='red')
ax2.spines['top'].set_visible(False)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=10)

ax1.set_title('10-Year Treasury Yield vs Tech Stocks Average Cumulative Return',
              fontsize=13, fontweight='bold')
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax1.xaxis.set_major_locator(mdates.YearLocator(2))
plt.setp(ax1.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 4. CPI vs Market Volatility

In [ ]:
# --- Average rolling_vol_30 across all tickers, resampled monthly ---
vol_daily = market_df[['date', 'rolling_vol_30']].copy()
vol_daily = vol_daily.sort_values('date')
vol_monthly = (
    vol_daily.set_index('date')
    .resample('ME')['rolling_vol_30']
    .mean()
    .reset_index()
    .rename(columns={'rolling_vol_30': 'avg_vol_30'})
)

# --- CPI monthly ---
cpi = (
    macro_df[macro_df['indicator_name'] == 'CPI (Inflation Proxy)']
    .sort_values('date')[['date', 'value']]
    .copy()
)
cpi_monthly = (
    cpi.set_index('date')
    .resample('ME')['value']
    .mean()
    .reset_index()
    .rename(columns={'value': 'cpi'})
)

# --- Merge ---
vol_monthly = vol_monthly.sort_values('date')
cpi_monthly = cpi_monthly.sort_values('date')
merged_cpi = pd.merge_asof(vol_monthly, cpi_monthly, on='date', direction='nearest')

# --- Plot dual-axis chart ---
fig, ax1 = plt.subplots(figsize=(14, 5))

ax1.plot(merged_cpi['date'], merged_cpi['avg_vol_30'],
         color='steelblue', linewidth=2, label='Avg 30-Day Rolling Volatility')
ax1.set_xlabel('Date', fontsize=11)
ax1.set_ylabel('Avg Rolling Volatility (30-Day)', color='steelblue', fontsize=11)
ax1.tick_params(axis='y', labelcolor='steelblue')
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

ax2 = ax1.twinx()
ax2.plot(merged_cpi['date'], merged_cpi['cpi'],
         color='red', linewidth=2, linestyle='--', label='CPI (Inflation Proxy)')
ax2.set_ylabel('CPI Value', color='red', fontsize=11)
ax2.tick_params(axis='y', labelcolor='red')
ax2.spines['top'].set_visible(False)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=10)

ax1.set_title('CPI vs Market Volatility', fontsize=13, fontweight='bold')
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax1.xaxis.set_major_locator(mdates.YearLocator(2))
plt.setp(ax1.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 5. Market Performance in High vs Low Rate Regimes

In [ ]:
# --- SPY daily returns resampled to monthly ---
spy_ret = market_df[market_df['ticker'] == 'SPY'][['date', 'daily_return']].copy()
spy_ret = spy_ret.sort_values('date')
spy_ret_monthly = (
    spy_ret.set_index('date')
    .resample('ME')['daily_return']
    .mean()
    .reset_index()
    .rename(columns={'daily_return': 'avg_daily_return'})
)

# --- Federal Funds Rate monthly (reuse fed_rate_monthly from Section 2) ---
# Merge SPY monthly returns with Fed Funds Rate
spy_ret_monthly = spy_ret_monthly.sort_values('date')
merged_regime = pd.merge_asof(spy_ret_monthly, fed_rate_monthly, on='date', direction='nearest')

# --- Label each month as High or Low rate regime ---
merged_regime['regime'] = merged_regime['fed_rate'].apply(
    lambda r: 'High Rate\n(Fed Funds >= 4%)' if r >= 4.0 else 'Low Rate\n(Fed Funds < 4%)'
)

# --- Group and compute average daily return per regime ---
regime_avg = (
    merged_regime.groupby('regime')['avg_daily_return']
    .mean()
    .reset_index()
    .rename(columns={'avg_daily_return': 'mean_daily_return'})
)
# Convert to percentage
regime_avg['mean_daily_return_pct'] = regime_avg['mean_daily_return'] * 100

print(regime_avg)

# --- Bar chart ---
fig, ax = plt.subplots(figsize=(7, 5))

colors = ['steelblue' if regime_avg['mean_daily_return_pct'].iloc[i] >= 0 else 'tomato'
          for i in range(len(regime_avg))]

bars = ax.bar(regime_avg['regime'], regime_avg['mean_daily_return_pct'],
              color=['steelblue', 'tomato'], width=0.4, edgecolor='white')

# Add value labels on bars
for bar, val in zip(bars, regime_avg['mean_daily_return_pct']):
    ypos = bar.get_height() + 0.001 if val >= 0 else bar.get_height() - 0.002
    ax.text(bar.get_x() + bar.get_width() / 2, ypos,
            f'{val:.4f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.axhline(0, color='black', linewidth=0.8, linestyle='-')
ax.set_title('SPY Avg Daily Return: High vs Low Rate Regimes', fontsize=13, fontweight='bold')
ax.set_xlabel('Interest Rate Regime', fontsize=11)
ax.set_ylabel('Avg Daily Return (%)', fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()